# SparkClient: Batch Job Lifecycle

This notebook demonstrates the end-to-end lifecycle of submitting, monitoring, and cleaning up a Spark batch job using `SparkClient` from the Kubeflow SDK.

### What you will learn:
1. Configuring and initializing `SparkClient`.
2. Submitting a remote `FileJob` with executor resource constraints.
3. Polling and waiting for execution completion.
4. Inspecting job metadata and driver pod status.
5. Listing active and completed Spark jobs.
6. Streaming driver logs for observability.
7. Cleaning up completed job resources.

## 1. Imports and Client Setup

Import the necessary classes from `kubeflow.spark` and configure your target Kubernetes namespace.

In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

# Namespace configuration (uses SPARK_TEST_NAMESPACE if set, else 'default')
namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)

# Initialize SparkClient
client = SparkClient(backend_config=backend_config)
print(f"SparkClient initialized for namespace: {namespace}")

## 2. Submit a Spark Batch Job

Submit a `FileJob` pointing to an executable Spark application script with custom CPU, memory, and shuffle partition configurations.

In [ ]:
REMOTE_JOB = "https://raw.githubusercontent.com/kubeflow/sdk/main/examples/spark/spark_job.py"

job_name = client.submit_job(
    job=FileJob(
        file_source=REMOTE_JOB,
        args=["10"],
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
    spark_conf={
        "spark.sql.shuffle.partitions": "8",
    },
)

print(f"Job submitted successfully: {job_name}")

## 3. Wait for Job Completion

Block execution until the Spark application reaches `COMPLETED` status or times out.

In [ ]:
print(f"Waiting for {job_name} to complete...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.COMPLETED},
    timeout=300,
)

print("Job completed successfully.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

## 4. Retrieve Job Metadata

Inspect the metadata associated with the submitted job.

In [ ]:
job_info = client.get_job(job_name)

print(f"Name: {job_info.name}")
print(f"Namespace: {job_info.namespace}")
print(f"Status: {job_info.status}")
print(f"Driver Pod: {job_info.driver_pod_name}")
print(f"Executors: {job_info.num_executors}")

## 5. List Spark Jobs

Query the cluster for all Spark jobs as well as jobs filtered by specific status states.

In [ ]:
jobs = client.list_jobs()
print(f"Found {len(jobs)} total Spark job(s).")
for j in jobs:
    print(f"- {j.name} | Status: {j.status} | Namespace: {j.namespace}")

completed_jobs = client.list_jobs(status={SparkJobStatus.COMPLETED})
print(f"\nFound {len(completed_jobs)} completed Spark job(s).")

## 6. Stream Driver Logs

Access logs directly from the Spark driver pod to observe output and diagnostics.

In [ ]:
print(f"Retrieving logs for {job_name} (first 20 lines):")
print("-" * 70)

for idx, line in enumerate(client.get_job_logs(job_name)):
    print(line.rstrip())
    if idx >= 20:
        print("...")
        break
print("-" * 70)

## 7. Clean Up Resources

Delete the completed Spark application resource from the cluster.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deletion requested successfully.")